<h3 style="color:#6FA8DC; font-weight:bold">03 — Normalization</h3>

<h5 style="color:#78B89A; font-weight:bold;">Min-Max Scaling, Mean Normalization, MaxAbs Scaling and Robust Scaling</h5>

In the previous notebook, we studied **Feature Scaling** and Standardisation.

Now we will study different scaling techniques that are commonly grouped under normalization or feature transformation.

In this notebook, we will cover:

- Min-Max Scaling — intuition and formula
- Min-Max Scaling — practical code example
- Mean Normalization
- MaxAbs Scaling
- Robust Scaling
- Normalization vs Standardization
- Visual comparison using the Wine dataset
- Important practical rules

We will use the provided `wine_data.csv` dataset so that the concepts remain connected to real Machine Learning workflows.

<div style="border-top:black 2px solid"></div>

# 1. What is Normalization?

Normalization is a technique used to transform numerical data into a more suitable scale.

Different features may have very different ranges.

For example:

| Feature | Approximate Range |
|---|---:|
| Alcohol | 11 to 15 |
| Magnesium | 70 to 160 |
| Proline | 278 to 1680 |

If an algorithm calculates distances, the feature with the larger numerical range may dominate the calculation.

Normalization reduces this problem by bringing features to a comparable scale.

### Simple definition

> **Normalization is the process of transforming numerical values into a common or controlled scale so that no feature dominates only because of its numerical magnitude.**

### Important note

In different books and courses, the word *normalization* may refer to different techniques.

It may mean:

- Min-Max Scaling
- Mean Normalization
- Vector Normalization

Therefore, always check the formula being used.

# 2. Why do we need Normalization?

Suppose we are working with a wine dataset.

The dataset contains features such as:

- Alcohol
- Malic acid
- Magnesium
- Proline

Consider two features:

```text
Alcohol  → approximately 11 to 15
Proline  → approximately 278 to 1680
```

If a distance-based algorithm calculates the distance between two wines, Proline may contribute much more simply because its values are larger.

### Algorithms that commonly benefit from normalization

- K-Nearest Neighbours
- K-Means Clustering
- DBSCAN
- Support Vector Machines
- Logistic Regression
- Neural Networks
- PCA

### Tree-based algorithms

Decision Trees and Random Forests generally do not require scaling in the same way because they split data using thresholds rather than relying directly on feature distances.

# 3. Types Covered in This Notebook

We will study the following techniques:

1. Min-Max Scaling
2. Mean Normalization
3. MaxAbs Scaling
4. Robust Scaling

We will also compare:

- Normalization
- Standardization

<div style="border-top:black 2px solid"></div>

# 4. Min-Max Scaling

## 4.1 Intuition

Min-Max Scaling converts values into a fixed range.

The most common range is:

```text
0 to 1
```

The smallest value becomes approximately `0`.

The largest value becomes approximately `1`.

All other values are placed proportionally between them.

### Example

Suppose the ages are:

```text
20, 30, 40, 50, 60
```

Minimum = 20

Maximum = 60

For age = 40:

```text
Scaled value = (40 - 20) / (60 - 20)
             = 20 / 40
             = 0.5
```

So, age 40 becomes `0.5`.

### Formula

\[
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
\]

Where:

- `x` = original value
- `xmin` = minimum value of the feature
- `xmax` = maximum value of the feature
- `x'` = scaled value

### Result

For the usual range:

```text
Minimum → 0
Maximum → 1


## 4.2 Min-Max Scaling diagram

```text
Original values

min ------------------------------ max
 20                                  60

After Min-Max Scaling

0 ----------------------------------- 1
```

The relative position of every value is preserved, but the numerical range changes.

## 4.3 Advantages of Min-Max Scaling

- Easy to understand.
- Produces a fixed range.
- Useful when a model expects bounded values.
- Maintains the relative ordering of values.
- Commonly used in some neural network workflows.

## 4.4 Limitations

- Sensitive to outliers.
- A single extreme minimum or maximum can compress most observations into a small range.
- It does not remove outliers.

## 4.5 Code Example — Min-Max Scaling

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

sample = pd.DataFrame({
    'Age': [20, 30, 40, 50, 60]
})

scaler = MinMaxScaler()

sample['Age_MinMax'] = scaler.fit_transform(sample[['Age']])

sample

### Important

`MinMaxScaler` expects a 2-dimensional input.

Therefore, we generally pass:

```python
df[['Age']]
```

instead of:

```python
df['Age']
```

because:

- `df['Age']` returns a Series.
- `df[['Age']]` returns a DataFrame.

<div style="border-top:black 2px solid"></div>

# 5. Loading the Wine Dataset

The provided dataset contains wine chemical measurements.

The first column is the class label.

The remaining columns are numerical chemical features.

For this notebook, we will use meaningful column names based on the classic Wine dataset.

In [ ]:
df = pd.read_csv('wine_data.csv', header=None)

df.columns = [
    'Class label',
    'Alcohol',
    'Malic acid',
    'Ash',
    'Alcalinity of ash',
    'Magnesium',
    'Total phenols',
    'Flavanoids',
    'Nonflavanoid phenols',
    'Proanthocyanins',
    'Color intensity',
    'Hue',
    'OD280/OD315',
    'Proline'
]

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

### Selecting features

We will use only two features initially so that the visual comparison is easy:

- Alcohol
- Malic acid

The class label is the target and should not be scaled as an input feature.

In [ ]:
X = df[['Alcohol', 'Malic acid']]
y = df['Class label']

X.head()

## 5.1 Original feature distributions

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 4))

sns.kdeplot(X['Alcohol'], fill=True, ax=ax1)
ax1.set_title('Alcohol Before Scaling')

sns.kdeplot(X['Malic acid'], fill=True, ax=ax2)
ax2.set_title('Malic Acid Before Scaling')

plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(
    data=df,
    x='Alcohol',
    y='Malic acid',
    hue='Class label',
    palette={1: 'red', 2: 'blue', 3: 'green'}
)

plt.title('Original Wine Data')
plt.show()

<div style="border-top:black 2px solid"></div>

# 6. Train-Test Split Before Scaling

In a Machine Learning project, we should split the dataset before learning scaling parameters.

Why?

A scaler learns values such as:

- Minimum
- Maximum
- Mean
- Median
- Standard deviation

These must be learned only from the training data.

Otherwise, information from the test set may leak into the training process.

### Correct workflow

```text
Raw Data
   ↓
Train-Test Split
   ↓
Fit scaler on X_train
   ↓
Transform X_train
   ↓
Transform X_test
```

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0,
    stratify=y
)

X_train.shape, X_test.shape

<div style="border-top:black 2px solid"></div>

# 7. Applying Min-Max Scaling to the Wine Dataset

In [ ]:
minmax_scaler = MinMaxScaler()

# Learn minimum and maximum only from training data
minmax_scaler.fit(X_train)

# Transform training and testing data
X_train_minmax = minmax_scaler.transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

X_train_minmax = pd.DataFrame(
    X_train_minmax,
    columns=X_train.columns,
    index=X_train.index
)

X_test_minmax = pd.DataFrame(
    X_test_minmax,
    columns=X_test.columns,
    index=X_test.index
)

X_train_minmax.head()

In [ ]:
np.round(X_train_minmax.describe(), 2)

### Observation

The training values are approximately between `0` and `1`.

The exact range of the test set may sometimes go slightly outside the training range if a test observation is smaller or larger than the training minimum or maximum.

This is normal because the scaler learned its parameters from training data only.

## 7.1 Before and After Min-Max Scaling

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.scatter(
    X_train['Alcohol'],
    X_train['Malic acid'],
    c=y_train
)
ax1.set_title('Before Min-Max Scaling')
ax1.set_xlabel('Alcohol')
ax1.set_ylabel('Malic acid')

ax2.scatter(
    X_train_minmax['Alcohol'],
    X_train_minmax['Malic acid'],
    c=y_train
)
ax2.set_title('After Min-Max Scaling')
ax2.set_xlabel('Scaled Alcohol')
ax2.set_ylabel('Scaled Malic acid')

plt.tight_layout()
plt.show()

### What changed?

- The numerical ranges changed.
- The relative pattern of the data remains similar.
- The features now have comparable ranges.
- The class structure is not intentionally changed by scaling.

<div style="border-top:black 2px solid"></div>

# 8. Mean Normalization

## 8.1 What is Mean Normalization?

Mean Normalization centers the data around zero and scales it using the range of the feature.

### Formula

\[
x' = \frac{x - \mu}{x_{max} - x_{min}}
\]

Where:

- `x` = original value
- `μ` = mean of the feature
- `xmax` = maximum value
- `xmin` = minimum value

### Intuition

First:

```text
Subtract the mean
```

This moves the center of the feature close to zero.

Then:

```text
Divide by the range
```

This controls the spread of the values.

### Result

Mean Normalization generally produces values around zero, but it does not guarantee:

- Minimum = 0
- Maximum = 1
- Standard deviation = 1

## 8.2 Manual Mean Normalization Example

In [ ]:
sample = pd.DataFrame({
    'Age': [20, 30, 40, 50, 60]
})

mean_normalized = (
    sample['Age'] - sample['Age'].mean()
) / (
    sample['Age'].max() - sample['Age'].min()
)

sample['Mean_Normalized_Age'] = mean_normalized

sample

### Interpretation

- Values below the mean become negative.
- Values above the mean become positive.
- The mean becomes approximately zero.
- The denominator is the feature range, not the standard deviation.

## 8.3 Mean Normalization on the Wine Dataset

In [ ]:
X_train_mean_norm = (
    X_train - X_train.mean()
) / (
    X_train.max() - X_train.min()
)

X_test_mean_norm = (
    X_test - X_train.mean()
) / (
    X_train.max() - X_train.min()
)

X_train_mean_norm.head()

In [ ]:
np.round(X_train_mean_norm.describe(), 2)

### Important practical rule

For the test set, use:

- Training mean
- Training minimum
- Training maximum

Do not calculate separate parameters from the test set.

<div style="border-top:black 2px solid"></div>

# 9. MaxAbs Scaling

## 9.1 What is MaxAbs Scaling?

MaxAbs Scaling divides every value by the maximum absolute value of that feature.

### Formula

\[
x' = \frac{x}{\max(|x|)}
\]

### Example

Suppose:

```text
[-10, -5, 0, 5, 10]
```

Maximum absolute value = 10.

After scaling:

```text
[-1, -0.5, 0, 0.5, 1]
```

### Properties

- Usually produces values between `-1` and `1`.
- Does not center the data around zero.
- Preserves zero values.
- Useful for sparse datasets.
- Does not require subtracting the mean.

### When is it useful?

It is useful when:

- The data contains many zeros.
- Sparse structure should be preserved.
- Centering the data is undesirable.

In [ ]:
from sklearn.preprocessing import MaxAbsScaler

maxabs_scaler = MaxAbsScaler()

X_train_maxabs = maxabs_scaler.fit_transform(X_train)
X_test_maxabs = maxabs_scaler.transform(X_test)

X_train_maxabs = pd.DataFrame(
    X_train_maxabs,
    columns=X_train.columns,
    index=X_train.index
)

X_test_maxabs = pd.DataFrame(
    X_test_maxabs,
    columns=X_test.columns,
    index=X_test.index
)

X_train_maxabs.head()

### MaxAbs Scaling vs Min-Max Scaling

| Min-Max Scaling | MaxAbs Scaling |
|---|---|
| Uses minimum and maximum | Uses maximum absolute value |
| Usually maps range to 0–1 | Usually maps values to -1–1 |
| Shifts the minimum to zero | Does not shift the data |
| May not preserve zero | Preserves zero values |

<div style="border-top:black 2px solid"></div>

# 10. Robust Scaling

## 10.1 What is Robust Scaling?

Robust Scaling uses:

- Median
- Interquartile Range (IQR)

instead of mean and standard deviation.

### Formula

\[
x' = \frac{x - Median}{IQR}
\]

where:

\[
IQR = Q3 - Q1
\]

### Why is it called robust?

Median and IQR are less affected by extreme values than mean and standard deviation.

### Real-life example

Consider customer spending:

```text
200, 250, 300, 350, 400, 100000
```

The last value may be a valid but extreme purchase.

Mean and standard deviation can be strongly influenced by this value.

Robust Scaling uses the median and IQR, so the extreme value has less influence on the scaling parameters.

In [ ]:
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler()

X_train_robust = robust_scaler.fit_transform(X_train)
X_test_robust = robust_scaler.transform(X_test)

X_train_robust = pd.DataFrame(
    X_train_robust,
    columns=X_train.columns,
    index=X_train.index
)

X_test_robust = pd.DataFrame(
    X_test_robust,
    columns=X_test.columns,
    index=X_test.index
)

X_train_robust.head()

## 10.2 Visual intuition for Robust Scaling

In [ ]:
outlier_example = pd.DataFrame({
    'Income': [25000, 28000, 30000, 32000, 35000, 500000]
})

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 4))

sns.boxplot(data=outlier_example, x='Income', ax=ax1)
ax1.set_title('Original Income Values')

robust_example = RobustScaler().fit_transform(
    outlier_example[['Income']]
)

sns.boxplot(x=robust_example[:, 0], ax=ax2)
ax2.set_title('After Robust Scaling')

plt.tight_layout()
plt.show()

### Important

Robust Scaling reduces the influence of outliers on the scaling parameters.

It does **not** delete outliers.

It also does not guarantee that all values will lie between 0 and 1.

<div style="border-top:black 2px solid"></div>

# 11. Normalization vs Standardization

These terms are frequently confused.

## Normalization

Normalization usually tries to bring values into a controlled range or representation.

Examples:

- Min-Max Scaling
- Mean Normalization
- MaxAbs Scaling
- Vector Normalization

## Standardization

Standardization uses:

\[
z = \frac{x - \mu}{\sigma}
\]

It produces:

- Mean approximately 0
- Standard deviation approximately 1

### Main difference

| Normalization | Standardization |
|---|---|
| Often controls the range | Controls mean and standard deviation |
| Min-Max commonly gives 0 to 1 | Gives z-scores |
| May use minimum and maximum | Uses mean and standard deviation |
| Useful when bounded values are desired | Useful for many statistical and ML algorithms |
| Sensitive to outliers depending on method | Mean and standard deviation are also affected by outliers |

## 11.1 Visual Comparison of Scaling Techniques

In [ ]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()

X_train_standard = standard_scaler.fit_transform(X_train)

X_train_standard = pd.DataFrame(
    X_train_standard,
    columns=X_train.columns,
    index=X_train.index
)

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 8))

sns.kdeplot(X_train['Alcohol'], fill=True, ax=axes[0, 0])
axes[0, 0].set_title('Original Alcohol')

sns.kdeplot(X_train_minmax['Alcohol'], fill=True, ax=axes[0, 1])
axes[0, 1].set_title('Min-Max Scaled Alcohol')

sns.kdeplot(X_train_mean_norm['Alcohol'], fill=True, ax=axes[1, 0])
axes[1, 0].set_title('Mean Normalized Alcohol')

sns.kdeplot(X_train_standard['Alcohol'], fill=True, ax=axes[1, 1])
axes[1, 1].set_title('Standardized Alcohol')

plt.tight_layout()
plt.show()

### What should we observe?

The distribution shape is generally preserved.

The main changes are:

- Location of the center
- Numerical spread
- Range of values
- Units of measurement

Scaling does not automatically make data normally distributed.

For example, Min-Max Scaling does not convert a skewed distribution into a Gaussian distribution.

<div style="border-top:black 2px solid"></div>

# 12. Comparison Table

| Technique | Formula / Idea | Main Result | Outlier Sensitivity |
|---|---|---|---|
| Min-Max Scaling | `(x - min) / (max - min)` | Usually 0 to 1 | High |
| Mean Normalization | `(x - mean) / (max - min)` | Centered around 0 | Moderate to high |
| MaxAbs Scaling | `x / max(abs(x))` | Usually -1 to 1 | High |
| Robust Scaling | `(x - median) / IQR` | Median-centered | Lower |
| Standardization | `(x - mean) / std` | Mean 0, std 1 | Moderate to high |

### Memory trick

- **MinMaxScaler** → Minimum and Maximum
- **Mean Normalization** → Mean and Range
- **MaxAbsScaler** → Maximum Absolute Value
- **RobustScaler** → Median and IQR
- **StandardScaler** → Mean and Standard Deviation

# 13. Which Technique Should We Choose?

### Choose Min-Max Scaling when:

- You need values in a fixed range.
- The data does not contain serious extreme values.
- A bounded input range is useful.

### Choose Mean Normalization when:

- You want to center values around the mean.
- You want to divide by the feature range.

### Choose MaxAbs Scaling when:

- Your data is sparse.
- You want to preserve zero values.
- You do not want to center the data.

### Choose Robust Scaling when:

- Your dataset contains significant outliers.
- Median and IQR are more reliable than mean and standard deviation.

### Choose Standardization when:

- You want z-scores.
- Your algorithm benefits from centered features.
- You are using algorithms such as Logistic Regression, SVM, KNN, or PCA.

<div style="border-top:black 2px solid"></div>

# 14. Important Practical Rules

1. Split the data before fitting the scaler.
2. Fit the scaler only on training data.
3. Transform the test data using the same fitted scaler.
4. Never fit a separate scaler on the test set.
5. Scaling does not remove outliers.
6. Scaling does not convert categorical data into numerical data.
7. Do not scale the target casually in classification problems.
8. Do not scale ID columns simply because they contain numbers.
9. Scaling changes numerical units, not the meaning of the feature.
10. Use a Pipeline in real projects to avoid data leakage.

### Correct workflow

```python
X_train, X_test, y_train, y_test = train_test_split(X, y)

scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

### Incorrect workflow

```python
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y
)
```

The incorrect workflow allows information from the test set to influence preprocessing.

# Final Revision

## Normalization

Normalization transforms numerical features into a comparable or controlled scale.

### Main techniques studied

1. Min-Max Scaling
2. Mean Normalization
3. MaxAbs Scaling
4. Robust Scaling

### Min-Max Formula

\[
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
\]

### Mean Normalization Formula

\[
x' = \frac{x - \mu}{x_{max} - x_{min}}
\]

### MaxAbs Formula

\[
x' = \frac{x}{\max(|x|)}
\]

### Robust Scaling Formula

\[
x' = \frac{x - Median}{IQR}
\]

### Standardization Formula

\[
z = \frac{x - \mu}{\sigma}
\]

### Final memory line

**Min-Max → 0 to 1**

**Mean Normalization → Center around mean**

**MaxAbs → Preserve zero and scale by maximum absolute value**

**Robust → Median and IQR**

**Standardization → Mean 0 and standard deviation 1**

> **Choose the scaling technique according to the data distribution, presence of outliers, algorithm, and desired numerical range.**